# Mission 04: Logging Middleware - 해답 노트북

이 노트북은 네 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [ ]:
# 1. 환경 로드
import sys
import os
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("app"))
load_dotenv(override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from app.tools import web_search

### [미션 1] LoggingMiddleware 구현하기

아래 코드는 `before_agent`, `after_agent`, `wrap_tool_call`을 구현하여 대화 세션 ID별로 로그 파일을 생성해 적재하는 솔루션 코드입니다.

In [ ]:
import time
import json
from typing import Any, Dict
from langchain.agents.middleware import AgentMiddleware

class StudentLoggingMiddleware(AgentMiddleware):
    def __init__(self, log_dir="./artifacts/logs"):
        self.log_dir = log_dir
        os.makedirs(self.log_dir, exist_ok=True)
        self._active_runs = {}
        
    def before_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        start_time = time.time()
        user_query = state.get("messages", [])[-1].content if state.get("messages") else "unknown"
        
        self._active_runs[id(runtime)] = {
            "start_time": start_time,
            "query": user_query
        }
        
        print(f"\n🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===")
        print(f"📥 사용자 질문: {user_query}")
        return None
        
    def after_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        run_data = self._active_runs.pop(id(runtime), {})
        start_time = run_data.get("start_time")
        duration_ms = int((time.time() - start_time) * 1000) if start_time else 0
        print(f"📤 에이전트 실행 완료 (소요: {duration_ms}ms)")
        
        user_query = run_data.get("query", "unknown")
        
        agent_response = ""
        dialogue_history = []
        messages = state.get("messages", [])
        if messages:
            agent_response = messages[-1].content
            for msg in messages:
                dialogue_history.append({
                    "role": msg.type,
                    "content": str(msg.content)
                })
                
        session_id = "unknown"
        if runtime and hasattr(runtime, "config") and runtime.config:
            session_id = runtime.config.get("configurable", {}).get("thread_id", "unknown")
            
        audit_log = {
            "event": "agent_execution",
            "session_id": session_id,
            "query": user_query,
            "response": agent_response,
            "dialogue_history": dialogue_history,
            "latency_ms": duration_ms,
            "status": "SUCCESS" if messages else "FAILED",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        
        self._append_log(session_id, audit_log)
        return None
        
    def wrap_tool_call(self, request, handler):
        logging_enabled = getattr(request.runtime.context, "logging_enabled", False) if request.runtime and request.runtime.context else False
        if not logging_enabled:
            return handler(request)
            
        tool_name = request.tool_call.get("name", "unknown_tool")
        tool_args = request.tool_call.get("args", {})
        start_time = time.time()
        
        print(f"🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: {tool_name}({tool_args})")
        
        response = handler(request)
        
        duration_ms = int((time.time() - start_time) * 1000)
        print(f"🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: {tool_name} (소요: {duration_ms}ms)")
        
        session_id = "unknown"
        if hasattr(request, "runtime") and hasattr(request.runtime, "config"):
            session_id = request.runtime.config.get("configurable", {}).get("thread_id", "unknown")
            
        tool_log = {
            "event": "tool_execution",
            "session_id": session_id,
            "tool_name": tool_name,
            "arguments": tool_args,
            "latency_ms": duration_ms,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        self._append_log(session_id, tool_log)
        return response
        
    def _append_log(self, session_id, log_data):
        log_file = os.path.join(self.log_dir, f"{session_id}.jsonl")
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_data, ensure_ascii=False) + "\n")

### [미션 2] 미들웨어가 연동된 에이전트 생성 및 검증

작성한 미들웨어를 create_agent에 등록하고 대화를 요청하여 로그 파일이 분리되어 저장되는지 검증합니다.

In [ ]:
log_directory = "./artifacts/logs"
thread_id = "session_logging_test_04"
log_filepath = os.path.join(log_directory, f"{thread_id}.jsonl")

if os.path.exists(log_filepath):
    os.remove(log_filepath)

logging_middleware = StudentLoggingMiddleware(log_dir=log_directory)
llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)

from app.prompts import CHATBOT_SYSTEM_PROMPT

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    middleware=[logging_middleware],
    context_schema=AgentContext
)

context_obj = AgentContext(logging_enabled=True)

res = agent.invoke(
    {"messages": [HumanMessage(content="마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)

### [미션 3] 생성된 감사 로그 검증

실제로 지정된 경로에 JSON 라인으로 로그들이 정상 저장되었는지 프린트해봅니다.

In [ ]:
if os.path.exists(log_filepath):
    print(f"📝 적재된 로그 내용 ({log_filepath}):")
    print("-" * 80)
    with open(log_filepath, "r", encoding="utf-8") as f:
        for line in f:
            print(line.strip())
    print("-" * 80)
else:
    print("❌ 로그 파일이 생성되지 않았습니다.")